# Interrogación 1 - Métricas de Recomendación



## Instrucciones Generales

1. Está prohibido el uso de IA como Chat GPT, Gemini, Claude o similares. Si se sorprende a alguien usándolo se le pedirá la prueba y se evaluará con 1,1.
2. En caso de utilizar Google Colab deben deshabilitar la *Asistencia con IA* (Ajustes > Asistencia con IA > Mostrar opciones de autocompletado de códigos basados en IA / Con permiso para usar funciones de IA generativa). Para quienes utilicen un IDE deben desactivar el autocompletado de herramientas de IA (Copilot, Claude Code, Codex, etc.).
3. Pueden realizar el práctico tanto en Google Colab como en un Jupyter Notebook local, pero no olvide entregar antes de que termine la hora de clases en este buzón de Canvas. Debe entregar únicamente el archivo main.ipynb completo.
4. Esta actividad es individual, está prohibido conversar con compañeros para realizarla.
5. Sí está permitido el uso de código de los prácticos realizados en las ayudantías, tanto el que se te entrega por el equipo docente como el que tú mismo/a has realizado.

Para realizar este práctico solo es necesario tener instalado Numpy y Pandas. Se sugiere tener instalada la última versión de Python y de Numpy.

La evaluación constará de una parte teórica y una segunda parte práctica donde deberás implementar con código en Python la métrica solicitada.

**NOMBRE ESTUDIANTE**: < colocar_nombre >

## Preguntas Teóricas (3pts)

En esta sección deberás contestar 3 preguntas teóricas

1. Describe la intuición detrás de la métrica **MRR (Mean Reciprocal Rank)** y **cómo se calcula**. ¿Qué tipo de tarea de recomendación evalúa mejor (por ejemplo, búsqueda, recomendación de un "ítem único", listas largas)? ¿Qué limitación tiene MRR para evaluar una la lista de recomendaciones?

**Respuesta**:

**Intuición y cálculo.** MRR (Mean Reciprocal Rank) mide qué tan arriba aparece el *primer* ítem relevante en la lista de recomendaciones. La intuición es simple: si el primer relevante aparece en la posición 1 el usuario está perfectamente servido (score = 1); si aparece en la posición 2 el score baja a 1/2; en la posición $k$ baja a $1/k$. Formalmente, para un conjunto de usuarios $U$:

$$\text{MRR} = \frac{1}{|U|} \sum_{u \in U} \frac{1}{\text{rank}_u}$$

donde $\text{rank}_u$ es la posición del primer ítem relevante en la lista recomendada al usuario $u$ (o $\infty$, aportando 0, si no aparece).

**Qué tarea evalúa mejor.** MRR es ideal para tareas en las que basta con encontrar *un* ítem relevante rápido: búsqueda (question answering, web search), recomendación de un "ítem único" (siguiente canción a reproducir, próxima película a ver, respuesta a una query). En esos escenarios el usuario típicamente consume el primer resultado útil y el resto importa poco.

**Limitación con múltiples relevantes.** MRR solo considera el primer acierto e ignora por completo al resto de ítems relevantes. Si un usuario tiene varios ítems relevantes, un recomendador que ubica uno en posición 1 y ninguno más en el top-k obtiene el mismo MRR que otro que pone relevantes en las posiciones 1, 2, 3, 4, aunque este último sea claramente mejor. Por eso en listas largas con múltiples relevantes conviene combinar MRR con métricas como MAP o nDCG.

2. Explica qué miden la **Diversidad (ILS de Ziegler)** y la **Novedad (self-information)**, cómo se calculan, y por qué son consideradas métricas *complementarias* a las métricas de accuracy (Precision, nDCG, etc.). ¿**Cómo afectan la Diversidad y la Novedad la percepción del usuario** sobre la calidad del recomendador (satisfacción, descubrimiento de nuevos ítems, efecto "filter bubble")? Da un ejemplo de un recomendador con alto nDCG pero baja novedad/diversidad y discute qué podría percibir el usuario frente a ese sistema.

**Respuesta**:

**Diversidad (ILS de Ziegler).** Mide qué tan distintos entre sí son los ítems recomendados a un usuario. Se calcula como la similitud intra-lista promedio entre todos los pares de ítems del top-k:

$$\text{ILS}(L) = \frac{1}{|L|(|L|-1)} \sum_{i \in L} \sum_{j \in L,\, j \neq i} \text{sim}(i, j)$$

donde $\text{sim}$ suele ser coseno sobre atributos/contenido de los ítems. **Valores bajos de ILS indican alta diversidad.**

**Novedad (self-information).** Mide qué tan "poco conocido" es lo que se recomienda. La auto-información de un ítem $i$ es $-\log_2 p(i)$, con $p(i)$ la fracción de usuarios que lo han consumido; ítems populares tienen baja novedad y la cola larga alta novedad. La novedad de la lista es el promedio sobre los ítems recomendados:

$$\text{Nov}(L) = \frac{1}{|L|} \sum_{i \in L} -\log_2 p(i)$$

**Por qué son complementarias a las métricas de accuracy.** Precision, nDCG, MAP, etc. solo miden si los ítems recomendados coinciden con los relevantes observados, pero son ciegas a *qué* se recomienda: un sistema puede maximizar accuracy recomendando siempre lo mismo (los hits más populares, o ítems muy parecidos entre sí). Diversidad y novedad capturan dimensiones ortogonales de la calidad percibida: variedad dentro de la lista y capacidad de descubrimiento más allá de lo obvio.

**Efecto en la percepción del usuario.** Listas diversas evitan la sensación de redundancia ("todo es lo mismo") y abren espacio a distintos intereses; alta novedad promueve el *descubrimiento* y sensación de que el sistema aporta valor frente a lo que el usuario ya conoce. Baja diversidad y baja novedad generan la *filter bubble*: el usuario ve cada vez más de lo mismo, se aburre y pierde confianza en que el sistema le muestre cosas nuevas.

**Ejemplo.** Un recomendador de películas que para un usuario fan de Marvel devuelve el top-10 con *Avengers*, *Avengers: Age of Ultron*, *Avengers: Infinity War*, *Avengers: Endgame*, *Iron Man 1/2/3*, *Captain America 1/2/3*. El nDCG será altísimo porque efectivamente son relevantes y bien rankeadas, pero la ILS será alta (muy similares entre sí → baja diversidad) y la novedad baja (son blockbusters conocidísimos). El usuario probablemente sienta que el sistema es "obvio" y no le está enseñando nada que no pudiera buscar solo; satisfacción a corto plazo razonable, pero baja retención y nula sensación de descubrimiento.

3. Explica con tus palabras en qué se diferencian **Precision** y **Recall** en el contexto de sistemas de recomendación, **tanto en la forma en que se calculan como en la forma de interpretar su resultado**. Luego, explica qué aporta **P@N (Precision at N)** respecto a la Precision global. ¿Por qué P@N suele ser más informativa al evaluar un ranker? Da un ejemplo donde dos sistemas tengan la misma Precision global pero un P@10 muy distinto.

**Respuesta:**

**Cálculo.** Sea $R_u$ el conjunto de ítems recomendados al usuario $u$ y $I_u$ el conjunto de ítems relevantes para $u$:

$$\text{Precision}(u) = \frac{|R_u \cap I_u|}{|R_u|} \qquad \text{Recall}(u) = \frac{|R_u \cap I_u|}{|I_u|}$$

Precision divide los aciertos por lo *recomendado*; Recall los divide por lo *relevante existente*.

**Interpretación.** Precision responde: "de lo que le mostré al usuario, ¿qué fracción le sirvió?" — mide pureza/ruido de la lista. Recall responde: "de todo lo que al usuario le habría gustado, ¿qué fracción alcancé a mostrarle?" — mide cobertura. Un recomendador puede tener Precision alta y Recall bajo (muestra pocos pero buenos ítems) o al revés (muestra todo, incluso ruido, asegurando cubrir los relevantes).

**Qué aporta P@N.** La Precision global se calcula sobre toda la lista recomendada, sin importar el orden ni cuántos ítems hay. P@N restringe el cálculo al top-N del ranking:

$$\text{P@N}(u) = \frac{|R_u^{1:N} \cap I_u|}{N}$$

Esto refleja mejor la experiencia real del usuario, que típicamente solo ve las primeras N posiciones (primera página de resultados, top-10 del feed). Por eso P@N es más informativa al evaluar un *ranker*: penaliza que los relevantes aparezcan tarde en la lista, mientras la Precision global los trata igual estén arriba o abajo.

**Ejemplo.** Dos sistemas recomiendan 100 ítems al usuario; ambos aciertan 20 relevantes → Precision global = 0.20 idéntica.

- Sistema A: los 20 relevantes aparecen en las posiciones 1–20 → P@10 = 10/10 = **1.0**.
- Sistema B: los 20 relevantes aparecen en las posiciones 81–100 → P@10 = 0/10 = **0.0**.

Para el usuario real (que mira el top-10), el sistema A es perfecto y el B es inservible, pese a tener la misma Precision global. P@N hace visible esa diferencia.

## Implementeación de Métrica (6 pts)

En esta sección deberás implementar la métrica **Zero Proportion @ k**, la cual mide la **fracción de usuarios para los cuales el recomendador no logra posicionar ningún ítem relevante dentro del top-k** de su lista de recomendaciones. Es una métrica útil para diagnosticar "fallos totales" del sistema (usuarios a los que no se les recomienda nada útil) y complementa a las métricas de ranking como AUC, nDCG o MAP.

Para hacer la evaluación vamos a utilizar el algoritmo de *collaborative filtering* denominado **Sapling Similarity**, presentado en el paper [Sapling Similarity: a performing and interpretable memory-based tool for recommendation](https://arxiv.org/abs/2210.07039), aplicado al dataset MovieLens 100K. Este dataset contiene 100.000 tuplas (usuario, item, ranking), de las cuales 80.000 se utilizarán para entrenamiento y 20.000 para testear. Los rankings se encuentran en un rango de 1 a 5.

Contarán con dos instancias del modelo ya entrenados: Basada en Usuario y Basada en Items. No deben modificar el código del modelo ni del entrenamiento.

In [ ]:
from sapling_similarity import SaplingSimilarity
import numpy as np

In [ ]:
# modelo basado en usuarios
sapling_similarity_users_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="user"
)

sapling_similarity_users_model.train()

# modelo basado en items
sapling_similarity_items_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="item"
)

sapling_similarity_items_model.train()

/content/sapling_similarity.py:129: RuntimeWarning: invalid value encountered in divide
  B = np.nan_to_num((1 - (co_ocurrences_matrix * (1 - co_ocurrences_matrix / items_interactions) + (items_interactions - co_ocurrences_matrix.T).T * (1 - (items_interactions - co_ocurrences_matrix.T).T / (
/content/sapling_similarity.py:130: RuntimeWarning: invalid value encountered in divide
  number_of_items - items_interactions))).T / (items_interactions * (1 - items_interactions / number_of_items))).T * np.sign(((co_ocurrences_matrix * number_of_items / items_interactions).T / items_interactions).T - 1))


Los modelos entrenados cuenta con el método `test`, el cual retorna dos elementos:
- El primer elemento corresponde a una lista de tuplas con los **ratings reales** para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_real).

- El segundo elemento corresponde a una lista de tuplas con los **ratings predichos** por el algoritmo para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_predicted).

In [ ]:
y_test, y_pred_users = sapling_similarity_users_model.test()
y_test, y_pred_items = sapling_similarity_items_model.test()

Además, los modelos entrenados cuentan con el método `predict`, que retorna para un `user_id` específico las lista de tuplas (item_id, ranking) de los items que el usuario no ha evaluado. Con estas predicciones es posible rankear una lista de items para un usuario ordenando descendente los items según el rating que predice el modelo.

In [ ]:
sapling_similarity_users_model.predict(user_id=1)[40:50]

[(1, np.int64(613), np.float64(4.300253965152188)),
 (1, np.int64(285), np.float64(4.293830614167815)),
 (1, np.int64(173), np.float64(4.290822452377635)),
 (1, np.int64(641), np.float64(4.2713802820914895)),
 (1, np.int64(1111), np.float64(4.26898936283368)),
 (1, np.int64(357), np.float64(4.265864353132638)),
 (1, np.int64(513), np.float64(4.262441584025546)),
 (1, np.int64(1431), np.float64(4.26102251250427)),
 (1, np.int64(657), np.float64(4.261015377303021)),
 (1, np.int64(479), np.float64(4.260759452916226))]

Se les entrega un diccionario con las predicciones hechas por cada una de las versiones del algoritmo, donde cada llave es un user_id y tiene como valor una lista con tuplas (item, rating), donde rating es el valor que el modelo predice que el usuario asignará al item. Los elementos para cada usuario estan ordenadas por el rating que predijo el modelo.

In [ ]:
user_based_predictions = {user_id: sapling_similarity_users_model.predict(user_id) for user_id in sapling_similarity_users_model.user_map.keys()}
item_based_predictions = {user_id: sapling_similarity_items_model.predict(user_id) for user_id in sapling_similarity_items_model.user_map.keys()}

1. Crea un diccionario con los items relevantes para cada usuario presentes en `y_test`. Cada llave del diccionario será un user_id y tendrá como valor un lista que contenga los item_ids de los items relevantes para ese usuario. Considere como relevante solo los items que se les asignó un **rating mayor o igual a 3**.

Ejemplo:
```python
{
  1: [3, 4, 8,...],
  2: [23, 24, 53,...],
  ...
}
```
donde [3, 4, 8,...] son los items relevantes para el usuario con ID = 1.

In [ ]:
user_relevant_items = {}

for user_id, item_id, rating in y_test:
    if rating >= 3:
        user_relevant_items.setdefault(user_id, []).append(item_id)


2. Completa la función `user_zero_hit`, la cual recibe como parámetros:

- `user_recommendations`: Lista de tuplas (item_id, rating_predicted) de un usuario, **ordenada descendentemente por rating predicho**.

- `user_relevant_items`: Lista de item_ids relevantes para el usuario.

- `k`: Largo de la lista de recomendación a considerar (top-k).

Esta función debe retornar **1** si **ninguno** de los primeros $k$ ítems recomendados se encuentra entre los relevantes del usuario, y **0** si al menos uno lo está. Formalmente:

$$
\text{zero\_hit}(u, k) =
\begin{cases}
1 & \text{si } \{i_1, i_2, \dots, i_k\} \cap I_u = \emptyset \\
0 & \text{en otro caso}
\end{cases}
$$

donde $\{i_1, \dots, i_k\}$ son los top-k ítems recomendados al usuario $u$ e $I_u$ es el conjunto de ítems relevantes para $u$.

---

**Ejemplo**

Supongamos que para un usuario tenemos:

`user_relevant_items = [A, B]`

`user_recommendations = [(C, 4.5), (D, 4.4), (E, 3.9), (A, 2.6)]`

- Con `k = 3` → top-3 = [C, D, E], intersección con [A, B] = ∅ → `zero_hit = 1` (fallo total en top-3).
- Con `k = 4` → top-4 = [C, D, E, A], intersección con [A, B] = {A} → `zero_hit = 0`.

In [ ]:
def user_zero_hit(user_recommendations, user_relevant_items, k):
    top_k_items = {item_id for item_id, _ in user_recommendations[:k]}
    relevant_set = set(user_relevant_items)
    return 0 if top_k_items & relevant_set else 1


3. Llama a la función `zero_proportion` para cada modelo y compara usando valores de k de 10, 100, 500, 1.000 y 2.000. Analice los resultados y mencione por qué podría producirse la diferencia de valores entre ambos modelos y entre distintos valores de k.

In [ ]:
def zero_proportion(predictions_dict, k):
    """
    Calcula la proporción de usuarios cuyo top-k no contiene ningún ítem relevante,
    dado un diccionario de predicciones y un tamaño k de recomendaciones.
    """
    zero_hits = []

    for user_id, predictions in predictions_dict.items():
        if user_id in user_relevant_items:
            # Convertir tripletas (user_id, item_id, rating) a pares (item_id, rating)
            rec_list = []
            for _, item_id, rating in predictions:
                rec_list.append((int(item_id), float(rating)))

            zero_hits.append(user_zero_hit(rec_list, user_relevant_items[user_id], k))

    return sum(zero_hits) / len(zero_hits) if zero_hits else 0.0


In [ ]:
k_values = [10, 100, 500, 1000, 2000]

print(f"{'k':>6} | {'User-based':>12} | {'Item-based':>12}")
print("-" * 40)
for k in k_values:
    zp_user = zero_proportion(user_based_predictions, k)
    zp_item = zero_proportion(item_based_predictions, k)
    print(f"{k:>6} | {zp_user:>12.4f} | {zp_item:>12.4f}")


     k |   User-based |   Item-based
----------------------------------------
    10 |       0.9877 |       0.6169
   100 |       0.1785 |       0.1723
   500 |       0.0077 |       0.0092
  1000 |       0.0000 |       0.0000
  2000 |       0.0000 |       0.0000


**Análisis de resultados.**

| k | User-based | Item-based |
|---:|---:|---:|
| 10 | 0.9877 | 0.6169 |
| 100 | 0.1785 | 0.1723 |
| 500 | 0.0077 | 0.0092 |
| 1000 | 0.0000 | 0.0000 |
| 2000 | 0.0000 | 0.0000 |

- **Efecto de $k$.** Como era esperable, Zero Proportion @ k es monótonamente decreciente en $k$: al ampliar la ventana del top-k aumentan las posiciones disponibles para que aparezca al menos un relevante, por lo que la fracción de usuarios con "fallo total" solo puede bajar. A $k=10$ la métrica es muy exigente (basta con que ningún relevante caiga en el top-10 para contar como fallo), a $k=100$ cae un orden de magnitud, y desde $k=500$ prácticamente todos los usuarios reciben al menos un relevante dentro del ranking. En $k=1.000$ y $k=2.000$ el valor es 0, lo que indica que el universo de ítems predichos por ambos modelos llega a cubrir los relevantes de test de *todos* los usuarios considerados — no hay usuarios cuyos relevantes estén totalmente fuera de la lista de predicción.

- **Diferencia user-based vs item-based.** La brecha más notoria está a **$k=10$**: el modelo user-based falla totalmente para el 98.77% de los usuarios, mientras que el item-based falla "solo" para el 61.69%. Es una diferencia enorme (≈37 puntos) que refleja que, en el top-10, el item-based es claramente mejor posicionando al menos un relevante arriba. La razón es característica de MovieLens 100K: hay menos usuarios (~943) que ítems (~1.682) pero las similitudes entre ítems son más densas y estables —cada ítem suele tener muchas evaluaciones— mientras que las similitudes entre usuarios son más ruidosas y sufren más con usuarios de historial escaso o gustos atípicos. Eso se traduce en que el user-based no logra "empujar" un relevante a las primerísimas posiciones para la mayoría de los usuarios.

- **Convergencia a $k=100$ y más allá.** A partir de $k=100$ ambos modelos son prácticamente indistinguibles (0.1785 vs 0.1723) e incluso el user-based es ligeramente mejor a $k=500$ (0.0077 vs 0.0092). Esto significa que **ambos modelos *conocen* a los ítems relevantes**, pero los rankean distinto: el item-based los sube antes y el user-based los deja más abajo en la lista. Cuando $k$ es lo suficientemente grande como para absorber esa diferencia de ranking, el fallo total desaparece de forma similar en ambos. Es un matiz importante: el item-based no es mejor porque "encuentre" más relevantes, sino porque los *rankea más arriba*.

- **Lectura diagnóstica.** Zero Proportion @ k aísla los *fallos totales*, complementando a métricas promedio como nDCG o MAP. La tabla sugiere que en producción con listas cortas (top-10, que es la ventana realista del usuario) conviene el **item-based**: casi 4 de cada 10 usuarios recibirían al menos un relevante arriba, frente a apenas 1 de cada 100 en user-based. Sin embargo, si el sistema trabajara con listas largas (top-100 o más, p.ej. ranking para re-ranking posterior o recuperación inicial) la diferencia entre ambos enfoques se difumina y la elección puede hacerse por otros criterios (costo, interpretabilidad, cold-start).

## Implementación de Métrica: nDCG con Relevancia Graduada (9 pts)

En esta sección deberás implementar la métrica **nDCG@k (Normalized Discounted Cumulative Gain)** usando los mismos modelos Sapling Similarity (user-based e item-based) sobre MovieLens 100K. A diferencia del nDCG clásico que trabaja con relevancia binaria, aquí se usará **relevancia graduada** derivada del rating real del set de test con el siguiente mapeo:

| rating real | $\text{rel}_i$ |
|---:|---:|
| < 3 | 0 |
| 3 | 1 |
| 4 | 2 |
| 5 | 3 |

De esta forma el valor de $\text{rel}_i$ en la fórmula de nDCG tomará valores en $\{0, 1, 2, 3\}$, lo que permite premiar más fuertemente el acierto de ítems que el usuario evaluó con rating alto (5) que los que evaluó con rating apenas positivo (3).

La fórmula para nDCG es la siguiente:

$$\text{DCG@k} = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i+1)} \qquad \text{nDCG@k} = \frac{\text{DCG@k}}{\text{iDCG@k}}$$

donde $\text{iDCG@k}$ es el DCG del ranking ideal.

1. Crea un diccionario `user_graded_relevance` con los ítems relevantes **graduados** para cada usuario presentes en `y_test`. Cada llave del diccionario será un `user_id` y tendrá como valor otro diccionario `{item_id: rel_grade}`, donde `rel_grade` es la relevancia graduada según la tabla anterior (solo se incluyen ítems con rating ≥ 3, es decir relevancia > 0).

Ejemplo:
```python
{
  1: {3: 2, 4: 3, 8: 1, ...},
  2: {23: 3, 24: 1, 53: 2, ...},
  ...
}
```
donde el usuario 1 evaluó al ítem 3 con rating 4 (rel = 2), al ítem 4 con rating 5 (rel = 3) y al ítem 8 con rating 3 (rel = 1).

In [ ]:
GRADE_MAP = {3: 1, 4: 2, 5: 3}

user_graded_relevance = {}
for user_id, item_id, rating in y_test:
    grade = GRADE_MAP.get(int(rating), 0)
    if grade > 0:
        user_graded_relevance.setdefault(user_id, {})[item_id] = grade


2. Completa la función `dcg_at_k`, la cual recibe como parámetros:

- `relevances`: Lista de **relevancias graduadas** (valores enteros en $\{0, 1, 2, 3\}$) de los ítems **en el mismo orden en que fueron rankeados por el modelo**. Es decir, `relevances[0]` es la relevancia graduada del ítem que el modelo puso en la posición 1 del ranking, `relevances[1]` la del ítem en posición 2, y así sucesivamente. Si el ítem de la posición $i$ no es relevante para el usuario (rating < 3 o no está en `user_graded_relevance`), en esa posición debe ir un 0.

    Ejemplo: si el modelo recomienda los ítems `[C, D, E, A, F]` en ese orden, y el usuario tiene `user_graded_relevance = {A: 3, B: 1, E: 2}`, entonces la lista a pasar es:
    ```python
    relevances = [0, 0, 2, 3, 0]
    ```
    (C no está → 0, D no está → 0, E tiene rel=2, A tiene rel=3, F no está → 0).

- `k`: Largo del top-k a considerar. La función debe usar únicamente las primeras $k$ posiciones de `relevances`.

La función debe retornar el DCG@k, definido como:

$$\text{DCG@k} = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i+1)}$$

donde $\text{rel}_i$ es la relevancia graduada en la posición $i$ del ranking (la posición 1 es la primera posición, por lo tanto el primer término tiene denominador $\log_2(2) = 1$).

In [ ]:
def dcg_at_k(relevances, k):
    relevances = relevances[:k]
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances))


3. Completa la función `user_ndcg`, la cual recibe como parámetros:

- `user_recommendations`: Lista de tuplas `(item_id, rating_predicted)` del usuario, **ordenada descendentemente por rating predicho**.
- `user_graded_relevance`: Diccionario `{item_id: rel_grade}` con las relevancias graduadas del usuario.
- `k`: Largo del top-k a considerar.

La función debe construir la secuencia de relevancias graduadas del top-k del ranking, calcular el DCG@k, calcular el iDCG@k, y retornar $\text{nDCG@k} = \text{DCG@k} / \text{iDCG@k}$.

In [ ]:
def user_ndcg(user_recommendations, user_graded_relevance, k):
    top_k_rels = [user_graded_relevance.get(item_id, 0)
                  for item_id, _ in user_recommendations[:k]]
    ideal_rels = sorted(user_graded_relevance.values(), reverse=True)[:k]

    idcg = dcg_at_k(ideal_rels, k)
    if idcg == 0:
        return 0.0
    return dcg_at_k(top_k_rels, k) / idcg


4. Llama a la función `mean_ndcg` para cada modelo y compara usando valores de k de 10, 100, 500, 1.000 y 2.000. Analice los resultados y mencione por qué podría producirse la diferencia de valores entre ambos modelos y entre distintos valores de k. Compare también contra lo observado con Zero Proportion @ k.

In [ ]:
def mean_ndcg(predictions_dict, k):
    """
    Calcula el nDCG@k promedio sobre todos los usuarios que tienen
    al menos un ítem relevante graduado en el set de testeo.
    """
    ndcgs = []

    for user_id, predictions in predictions_dict.items():
        if user_id not in user_graded_relevance:
            continue
        # Convertir tripletas (user_id, item_id, rating) a pares (item_id, rating)
        rec_list = [(int(item_id), float(rating)) for _, item_id, rating in predictions]
        ndcgs.append(user_ndcg(rec_list, user_graded_relevance[user_id], k))

    return sum(ndcgs) / len(ndcgs) if ndcgs else 0.0


k_values = [10, 100, 500, 1000, 2000]

print(f"{'k':>6} | {'User-based':>12} | {'Item-based':>12}")
print("-" * 40)
for k in k_values:
    ndcg_user = mean_ndcg(user_based_predictions, k)
    ndcg_item = mean_ndcg(item_based_predictions, k)
    print(f"{k:>6} | {ndcg_user:>12.4f} | {ndcg_item:>12.4f}")


     k |   User-based |   Item-based
----------------------------------------
    10 |       0.0008 |       0.0744
   100 |       0.0852 |       0.1500
   500 |       0.2519 |       0.3074
  1000 |       0.3109 |       0.3497
  2000 |       0.3204 |       0.3647


**Análisis de resultados.**

| k | User-based | Item-based |
|---:|---:|---:|
| 10 | 0.0008 | 0.0744 |
| 100 | 0.0852 | 0.1500 |
| 500 | 0.2519 | 0.3074 |
| 1000 | 0.3109 | 0.3497 |
| 2000 | 0.3204 | 0.3647 |

- **Efecto de $k$.** Como era esperable, nDCG@k es monótonamente creciente en $k$ para ambos modelos: al ampliar la ventana aparecen más relevantes y el DCG del numerador sube, mientras el IDCG también crece pero acotado por la cantidad real de relevantes del usuario. El salto más grande ocurre entre $k=10$ y $k=500$ (el user-based pasa de 0.0008 a 0.2519, el item-based de 0.0744 a 0.3074) y luego la curva se aplana: entre $k=1000$ y $k=2000$ el user-based apenas sube de 0.3109 a 0.3204 y el item-based de 0.3497 a 0.3647. Eso indica que una vez que $k$ supera la cantidad típica de relevantes por usuario, agregar posiciones adicionales aporta poco al ranking útil.

- **Diferencia user-based vs item-based.** El item-based domina en **todos** los valores de $k$, pero la brecha relativa más brutal está a **$k=10$**: 0.0744 vs 0.0008, un factor ~90×. Esto es consistente con lo observado en Zero Proportion @ k: el user-based casi nunca logra colocar un relevante en el top-10 (Zero Proportion 0.9877), por lo que su DCG@10 es prácticamente 0 y su nDCG también. El item-based, en cambio, sí ubica relevantes arriba para ~38% de los usuarios y eso se refleja directamente en el nDCG@10. A medida que $k$ crece la brecha absoluta se achica (de 0.0736 en k=10 a 0.0443 en k=2000) pero el item-based se mantiene consistentemente superior.

- **Valores absolutos modestos.** Incluso en el mejor caso (item-based a $k=2000$) el nDCG llega a 0.3647, no a valores cercanos a 1. Esto no significa que los modelos sean malos, sino que están ordenando un espacio de ~1.600 ítems donde solo ~20 son relevantes por usuario: es muy difícil que los relevantes caigan *exactamente* en las primeras posiciones. La relevancia graduada también hace más exigente la métrica — para alcanzar nDCG=1 el modelo tendría que poner los rel=3 estrictamente antes que los rel=2 y estos antes que los rel=1, no basta con separar relevantes de irrelevantes.

- **Comparación con Zero Proportion.** Zero Proportion @ 10 ya nos decía que user-based ≫ item-based en fallos totales. nDCG@10 refina ese diagnóstico: no solo el user-based falla en colocar relevantes, sino que además, incluso cuando *no* falla totalmente, los coloca en posiciones tan bajas del top-10 que su contribución al DCG es despreciable. A $k=500$, donde Zero Proportion era muy similar entre modelos (0.0077 vs 0.0092), el nDCG sigue mostrando una brecha de ~5 puntos a favor del item-based: Zero Proportion ya no los distingue (casi nadie falla totalmente), pero nDCG sí distingue *qué tan bien* los rankean, confirmando que el item-based no solo encuentra relevantes antes sino que los ordena mejor dentro del top.

- **Conclusión.** Si hay que elegir un modelo para listas cortas (top-10/100) la decisión es clara: item-based. Para listas largas (top-500 o más) el item-based sigue siendo mejor pero la brecha se reduce y otros criterios (costo computacional, cold-start, interpretabilidad) pueden pesar más en la elección final.